# P1-28 · Experiment E3: training on expert-corrected anatomical masks (mini-MIAS, 5-fold CV) — session A (T4 GPU; run A and B at the same time)
Two copies (A and B) share the 100 runs (4 variants x 5 seeds x 5 folds) through lock files. Resumable: after a disconnect, delete the runtime, reconnect and **Run all**.
Run P1-26b and P1-27b first. Read-only for `QSEG_FINAL`, `e3m` and `e3_refs`; writes only `MyDrive/P1_v2/e3_runs/`.
The prespecified text (cell 1) is stored with a UTC time stamp before the first run.

In [ ]:
# 1 · Settings, Drive, protocol record
SESSION = 'A'
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, json, time, shutil, random, gc
import numpy as np, pandas as pd
ROOT = '/content/drive/MyDrive/QSEG_FINAL'          # read only
OUT  = '/content/drive/MyDrive/P1_v2'
E3   = f'{OUT}/e3m'; REFS = f'{OUT}/e3_refs'; RC = f'{OUT}/revision_checks'; RUNS_DIR = f'{OUT}/e3_runs'
LOCAL1 = '/content/qseg_local'; LOCAL3 = '/content/e3_local'; TMP = '/content/e3_tmp'
for d in (RUNS_DIR, TMP): os.makedirs(d, exist_ok=True)
assert os.path.exists(E3 + '/data/manifest.csv'), 'Run notebook P1-27b first'
PROTO = """Paper 1 · experiment E3 (training on expert-corrected anatomical masks), written before any of these runs was trained.

Motivation: all earlier networks were trained on rule-based (proxy) masks of one private hospital set, so the null result might reflect
the easy proxy target or the data set rather than the circuit.
Training data: public mini-MIAS (322 digitized screen-film mammograms of 161 women) with the expert-corrected breast and pectoral-muscle
masks of Verboom et al. (Zenodo 10.5281/zenodo.10149914), in the orientation verified visually (all 322 masks: transpose of the stored
array). Target: breast-plus-pectoral foreground (labels 1 and 2), the region that the labelling rule targets. Images and masks as in
the external evaluation of the paper (256 x 256). Five-fold cross-validation by woman (women shuffled once with numpy RandomState(42),
dealt round-robin to five folds); for fold k the test set is fold k, the validation set (early stopping) fold k+1 (mod 5) and the
training set the other three folds, so every image is tested exactly once per seed and no woman contributes to two roles.
Models: HQB-UNet, HQB-cls, HQB-trig, HQB-free with skip connections (architectures of E1, unchanged); seeds 0-4 in each fold
(4 variants x 5 seeds x 5 folds = 100 runs). Matched initialization: for each fold and seed, every variant first builds its own model
(so its four-to-four mapping is initialized by the seed), then all layers it shares with the HQB-UNet are overwritten with the initial
weights of the HQB-UNet built with the same seed; the equality of all shared tensors is verified and recorded for every run. Data order,
augmentation and dropout are fixed by the seed.
Training and checkpoint: exactly as E1 (Adam 1e-4, batch 16, BCE + soft Dice, fp32, Paper 1 augmentation and seed handling,
up to 200 epochs, learning rate halved after 4 epochs without improvement of the training loss (min 1e-7), early stopping on the
validation loss from epoch 15 with patience 10, lowest-validation-loss weights tested).
Primary estimands: HQB-UNet - HQB-cls, HQB-UNet - HQB-trig and HQB-UNet - HQB-free in mean per-image Dice against the expert
breast-plus-pectoral masks over all 322 images, each image scored by the model of the fold in which it was tested (out-of-fold).
Uncertainty: crossed hierarchical bootstrap over women and seeds (10,000 replicates, rng(0)), 95% percentile CI and two-sided
bootstrap P; Holm over the three contrasts. Each fold is also reported separately. Pre-specified interpretation: a contrast is called
resolved if its Holm-adjusted P < 0.05. Secondary: equivalence within +/-0.01 per-image Dice is concluded if the 90% CI lies inside
+/-0.01.
Secondary (external, no fine-tuning): the same contrasts against the manual outlines of the 84 hospital test images (reader 1; reader 2
on 20 images; resampling patients), breast-plus-pectoral (matching the target) and breast-only; against the expert breast-only region
of mini-MIAS; code replacement at test time (constant and swapped codes); finite-shot readout of the HQB-UNet; seed-sensitivity checks
(independent seed resampling, t interval over seed-level differences, leave one seed out).
All runs are reported; none is discarded. Weights are deleted after evaluation; probability maps (uint8) are kept."""
proto = OUT + '/PROTOCOL_e3_prespecified.txt'
if not os.path.exists(proto):
    open(proto, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
print(open(proto).read().splitlines()[-1])
REF = 'full'          # reference region matching the training target

In [ ]:
# 2 · Data: E3 (mini-MIAS, local copy), hospital test set (Paper 1 data), reader masks
if not os.path.exists(LOCAL3 + '/data/manifest.csv'): shutil.copytree(E3 + '/data', LOCAL3 + '/data')
if not os.path.exists(LOCAL1 + '/data/manifest.csv'):
    shutil.copytree(ROOT + '/results/data', LOCAL1 + '/data', ignore=shutil.ignore_patterns('*.npy', '*.h5'))
sys.path.insert(0, ROOT + '/code')
import tensorflow as tf
from tensorflow.keras import layers, models
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
from qseg.config import Config
from qseg.models import conv_block, output_layer, TFCircuitLayer
from qseg.datasets import load_manifest, make_dataset, load_arrays
from qseg.evaluation import per_image_overlap
from qseg.train import bce_dice_loss, soft_dice
cfg = Config(work_dir=LOCAL3); man0 = load_manifest(cfg)
MAN = {k: man0.assign(split=man0[f'split{k}']) for k in range(5)}
X3 = {(k, s): load_arrays(cfg, s, MAN[k]) for k in range(5) for s in ('train', 'test')}
cfg1 = Config(work_dir=LOCAL1); H = load_arrays(cfg1, 'test', load_manifest(cfg1))     # 84 hospital test images and proxy masks
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
hid = list(H[2].file_id)
RD = {}
for rd in ('reader1', 'reader2'):
    for k in ('full', 'breast'):
        have = [i for i, f in enumerate(hid) if os.path.exists(f'{REFS}/{rd}/{k}/{f}.png')]
        RD[(rd, k)] = (have, np.stack([_png(f'{REFS}/{rd}/{k}/{hid[i]}.png', True) for i in have]))
BREAST = {f: _png(f'{RC}/masks_final/breast/{f}.png', True) for f in man0.file_id}      # expert breast-only region (secondary)
print({k: len(v[0]) for k, v in X3.items()}, '| val', [int((MAN[k].split == 'val').sum()) for k in range(5)], '| hospital test', len(hid),
      '| reader masks', {k: len(v[0]) for k, v in RD.items()}, '| TF', tf.__version__)

In [ ]:
# 3 · Models (Paper 1 hybrid architecture; optional removal of the skip connections) and evaluation helpers
class CodeSwitch(layers.Layer):
    """Identity during training; at test time can replace the four-value code by supplied codes."""
    def build(self, input_shape):
        self.on = self.add_weight(shape=(), initializer='zeros', trainable=False, name='on')
        self.buf = self.add_weight(shape=(cfg.batch_size, 4), initializer='zeros', trainable=False, name='buf')
    def call(self, z):
        n = tf.shape(z)[0]
        return z * (1.0 - self.on) + self.buf[:n] * self.on

class TrigLayer(layers.Layer):
    """20-parameter trigonometric 4 -> 4 map: z = tanh(A cos((pi/2) x + phi))."""
    def build(self, input_shape):
        self.A = self.add_weight(shape=(4, 4), initializer='glorot_uniform', name='A')
        self.phi = self.add_weight(shape=(4,), initializer=tf.keras.initializers.RandomUniform(0.0, 2 * np.pi), name='phi')
    def call(self, x):
        return tf.tanh(tf.matmul(tf.cos(np.pi / 2 * x + self.phi), self.A, transpose_b=True))

def build(kind, skips=True, filters=(32, 64, 128, 256, 512), dropouts=(0.1, 0.1, 0.2, 0.2, 0.3)):
    inp = layers.Input((256, 256, 1), name='input_image')
    sk, x = [], inp
    for i, (f, d) in enumerate(zip(filters[:-1], dropouts[:-1])):
        x = conv_block(x, f, d, name=f'enc{i + 1}'); sk.append(x)
        x = layers.MaxPooling2D(2, name=f'pool{i + 1}')(x)
    x = conv_block(x, filters[-1], dropouts[-1], name='bottleneck')
    z = layers.Dense(4, activation='tanh', name='pre_dense')(layers.GlobalAveragePooling2D(name='gap')(x))
    if kind == 'quantum':   z = TFCircuitLayer(4, 2, name='quantum_circuit')(z)
    elif kind == 'trig':    z = TrigLayer(name='trig_replacement')(z)
    elif kind == 'classical': z = layers.Dense(4, activation='tanh', name='classical_replacement')(z)
    z = CodeSwitch(name='code_switch')(z)
    q = layers.Dense(512, activation='relu', name='post_dense')(z)
    x = layers.Reshape((16, 16, filters[-1]), name='q_reshape')(
        layers.Dense(16 * 16 * filters[-1], activation='relu', name='q_expand')(q))
    for i, (f, s, d) in enumerate(zip(reversed(filters[:-1]), reversed(sk), reversed(dropouts[:-1]))):
        x = layers.UpSampling2D(2, name=f'up{i + 1}')(x)
        if skips: x = layers.Concatenate(name=f'concat{i + 1}')([x, s])
        x = conv_block(x, f, d, name=f'dec{i + 1}')
    return models.Model(inp, output_layer(x), name=f'HQB_{kind}_{"skip" if skips else "noskip"}')

def set_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def predict(model, x, codes=None):
    sw = model.get_layer('code_switch'); out = []
    for a in range(0, len(x), cfg.batch_size):
        xb = x[a:a + cfg.batch_size]
        if codes is None: sw.on.assign(0.0)
        else:
            buf = np.zeros((cfg.batch_size, 4), np.float32); buf[:len(xb)] = codes[a:a + len(xb)]
            sw.buf.assign(buf); sw.on.assign(1.0)
        out.append(model(xb, training=False).numpy()[..., 0])
    sw.on.assign(0.0)
    return np.concatenate(out)

def code_model(model):
    return tf.keras.Model(model.input, model.get_layer('code_switch').input)

def dice(Y, p):
    return per_image_overlap((Y[..., 0] > 0.5).astype(np.uint8), (p >= cfg.threshold).astype(np.uint8))[:, 0]

def shots(z, n, rng):
    """Finite-shot estimate of <Z> per qubit: p(+1) = (1 + z)/2, n shots."""
    p = np.clip((1 + z) / 2, 0, 1)
    return 2 * rng.binomial(n, p) / n - 1

def evaluate(model, kind, x, Y, z_tr):
    cm = code_model(model)
    z = cm.predict(x, batch_size=cfg.batch_size, verbose=0)
    d0 = dice(Y, predict(model, x))
    r = pd.DataFrame({'dice': d0})
    r['dice_check_switch'] = dice(Y, predict(model, x, z))             # must equal dice (sanity check)
    r['drop_const'] = d0 - dice(Y, predict(model, x, np.repeat(z_tr.mean(0)[None], len(x), 0)))
    for k in range(5):
        idx = np.random.default_rng(k).integers(0, len(z_tr), len(x))
        r[f'drop_swap{k}'] = d0 - dice(Y, predict(model, x, z_tr[idx]))
    r['drop_swapavg'] = r[[f'drop_swap{k}' for k in range(5)]].mean(1)
    if kind == 'quantum':
        for n in (100, 1000, 10000):
            ds = [dice(Y, predict(model, x, shots(z, n, np.random.default_rng(100 + k)))) for k in range(5)]
            r[f'dice_shots{n}'] = np.mean(ds, 0)
    r['code_sd'] = float(z.std(0).mean())
    return r


In [ ]:
# 4 · Training and evaluation of all runs (resumable; two sessions share the work through lock files)
KIND = {'hq': 'quantum', 'hc': 'classical', 'hf': 'none', 'ht': 'trig'}
RUNS = [(f'E3f{k}_{m}', KIND[m], k, s) for s in range(5) for k in range(5) for m in ('hq', 'hc', 'ht', 'hf')]
LOCK_H = 4
def claim(d):
    if os.path.exists(d + '/run.json'): return False
    lk = d + '/.lock'
    if os.path.exists(lk):
        who, t = open(lk).read().split()
        if who != SESSION and time.time() - float(t) < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}')
    time.sleep(20)
    return open(lk).read().split()[0] == SESSION

def run(name, kind, split, seed):
    d = f'{RUNS_DIR}/{name}_seed{seed}'
    if not claim(d): return
    man = MAN[split]
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.mixed_precision.set_global_policy('float32'); set_seeds(seed)
    tr, _ = make_dataset(cfg, 'train', seed, man); va, _ = make_dataset(cfg, 'val', seed, man)
    model = build(kind, True)
    # matched initialization: all layers shared with the HQB-UNet get the HQB-UNet initial weights of the same seed
    set_seeds(seed); ref = build('quantum', True); refw = {l.name: l.get_weights() for l in ref.layers if l.weights}
    shared = [l for l in model.layers if l.weights and l.name in refw and l.name not in ('quantum_circuit',)]
    for l in shared: l.set_weights(refw[l.name])
    init_check = max(float(np.abs(a - b).max()) for l in shared for a, b in zip(l.get_weights(), refw[l.name]))
    n_shared = int(sum(w.size for l in shared for w in l.get_weights())); del ref, refw; gc.collect()
    set_seeds(seed)
    model.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate), loss=bce_dice_loss, metrics=[soft_dice], jit_compile=False)
    ck = f'{TMP}/{name}_{seed}.weights.h5'
    cbs = [tf.keras.callbacks.TerminateOnNaN(),
           tf.keras.callbacks.ModelCheckpoint(ck, monitor='val_loss', save_best_only=True, save_weights_only=True),
           tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, start_from_epoch=15),
           tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=4, min_lr=1e-7),
           tf.keras.callbacks.CSVLogger(d + '/history.csv')]
    t0 = time.time(); h = model.fit(tr, validation_data=va, epochs=200, callbacks=cbs, verbose=0); secs = time.time() - t0
    model.load_weights(ck)
    z_tr = code_model(model).predict(X3[(split, 'train')][0], batch_size=cfg.batch_size, verbose=0)
    xt, yt, dft = X3[(split, 'test')]
    te = evaluate(model, kind, xt, yt, z_tr); te.insert(0, 'file_id', dft.file_id.values); te.insert(1, 'patient_id', dft.patient_id.values)
    pt = predict(model, xt)
    te['dice_breast_only'] = dice(np.stack([BREAST[f] for f in dft.file_id]), pt)
    ph = predict(model, H[0])
    ho = pd.DataFrame({'file_id': hid, 'patient_id': H[2].patient_id.values, 'dice_proxy': dice(H[1], ph)})
    for (rd, k), (idx, Y) in RD.items():
        col = np.full(len(hid), np.nan); col[idx] = dice(Y, ph[idx]); ho[f'dice_{rd}_{k}'] = col
    te.to_csv(d + '/test_per_image.csv', index=False); ho.to_csv(d + '/hospital_per_image.csv', index=False)
    np.savez_compressed(d + '/prob_uint8.npz', test=(pt * 255).round().astype(np.uint8), test_file_id=dft.file_id.values,
                        hospital=(ph * 255).round().astype(np.uint8))
    hist = pd.DataFrame(h.history); vl = hist['val_loss'].to_numpy()
    lr = hist['learning_rate'] if 'learning_rate' in hist else hist.get('lr', pd.Series([np.nan]))
    r = dict(name=name, kind=kind, fold=split, seed=seed, init_shared_params=n_shared, init_shared_max_abs_diff=init_check, epochs_run=len(vl), best_epoch=int(np.nanargmin(vl)) + 1, final_lr=float(lr.iloc[-1]),
             minutes=round(secs / 60, 1), params=int(model.count_params()), test_dice=float(te.dice.mean()),
             test_dice_breast_only=float(te.dice_breast_only.mean()),
             hospital_reader1_full=float(ho.dice_reader1_full.mean()), hospital_reader1_breast=float(ho.dice_reader1_breast.mean()),
             hospital_proxy=float(ho.dice_proxy.mean()), test_drop_const=float(te.drop_const.mean()), test_drop_swapavg=float(te.drop_swapavg.mean()),
             switch_check_max=float(np.abs(te.dice - te.dice_check_switch).max()), session=SESSION, tf_version=tf.__version__)
    json.dump(r, open(d + '/run.json', 'w'), indent=2)
    os.remove(ck); del model; tf.keras.backend.clear_session(); gc.collect()
    done = sum(os.path.exists(f'{RUNS_DIR}/{a}_seed{s}/run.json') for a, _, _, s in RUNS)
    print(time.strftime('%H:%M UTC', time.gmtime()), f"| {name} seed {seed}: {r['epochs_run']} ep (best {r['best_epoch']}), {r['minutes']} min | "
          f"test {r['test_dice']:.4f} (breast only {r['test_dice_breast_only']:.4f}) | "
          f"reader1 full {r['hospital_reader1_full']:.4f} breast {r['hospital_reader1_breast']:.4f} | init check {init_check:.1e} | finished {done} / {len(RUNS)}", flush=True)

order = RUNS if SESSION == 'A' else RUNS[::-1]
for args in order:
    run(*args)
left = [a for a in RUNS if not os.path.exists(f'{RUNS_DIR}/{a[0]}_seed{a[3]}/run.json')]
print(f'session {SESSION} has nothing left to start; {len(left)} run(s) still running in the other session')

In [ ]:
# 5 · Overview of finished runs
rows = [json.load(open(f'{RUNS_DIR}/{a}_seed{s}/run.json')) for a, _, _, s in RUNS if os.path.exists(f'{RUNS_DIR}/{a}_seed{s}/run.json')]
ov = pd.DataFrame(rows); ov.to_csv(RUNS_DIR + '/overview.csv', index=False)
print(ov.groupby('name')[['init_shared_max_abs_diff', 'test_dice', 'test_dice_breast_only', 'hospital_reader1_full', 'hospital_reader1_breast', 'hospital_proxy',
                          'test_drop_const', 'test_drop_swapavg', 'epochs_run', 'best_epoch']].mean().round(4).to_string())

In [ ]:
# 6 · Auto-shutdown
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()